# Experiment: Feynman Teacher Tuning Playground

Objective:
- Tune Feynman teacher training and post-prune refit parameters.
- Keep benchmark export format, plotting, and teacher cache behavior unchanged.

Scope:
- Focus on `feynman_I_12_1` and `feynman_I_12_4` first.
- Use the existing `scripts.icbr_benchmark` CLI so results are directly comparable.

In [1]:
from __future__ import annotations

import csv
import gc
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

try:
    import torch
except Exception:
    torch = None


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Cannot locate repo root from notebook path.")


NOTEBOOK_PATH = Path.cwd()
REPO_ROOT = find_repo_root(NOTEBOOK_PATH)
PYTHON_EXE = Path(sys.executable)

print(f"REPO_ROOT={REPO_ROOT}")
print(f"PYTHON_EXE={PYTHON_EXE}")

REPO_ROOT=d:\chenpeng\Documents\学习\数学\毕业论文\symkan-experiments
PYTHON_EXE=c:\Users\chenpeng\miniconda3\envs\kan\python.exe


## Tuning Config

Adjust this cell first. The defaults below are set for your current quality-oriented direction:
- `train_num=2000`, `test_num=1000`
- `train_steps=200`
- sparse regularization `lamb=1e-3`
- post-prune refit params can be tuned independently.

In [2]:
CONFIG = {
    "profile": "feynman_reference",
    "tasks": ["feynman_I_12_4"],
    #"tasks": ["feynman_I_12_1", "feynman_I_12_4","feynman_I_8_14"],
    "seeds": [1],
    "feynman_root": "datasets",
    "feynman_variant": "Feynman_with_units",
    "feynman_fit_opt": "Adam",
    "train_num": 2000,
    "test_num": 1000,
    "train_steps": 200,
    "lr": 1e-2,
    "lamb": 1e-2,
    "prune_iters": 3,
    "teacher_cache_mode": "off",
    "teacher_cache_version": "feynman_tune_v1",
    "teacher_cache_dir": "outputs/teacher_cache_feynman_tune",
    "output_dir": "outputs/icbr_benchmark_feynman_tune_example",
    "feynman_width_mid": "[5,2]",
    "feynman_split_strategy": "random",
    "feynman_split_strategy_seed": 1,
    "feynman_dataset_select_seed": 1,
    "feynman_post_prune_steps": 100,
    "feynman_post_prune_lr": 1e-3,
    "feynman_post_prune_lamb": 1e-3,
    "feynman_post_prune_eval_every": 5,
    "feynman_post_prune_min_delta": 1e-6,
    "feynman_post_prune_patience": 2,
    "teacher_max_test_mse": 0.1,
    "teacher_min_test_r2": 0.0,
    "quiet": False,
}

#CONFIG

## Run Benchmark Once

This runs `python -m scripts.icbr_benchmark` with the current config.
It keeps rows/summary/report exports and uses persistent teacher cache.

In [ ]:
def build_benchmark_command(cfg: dict[str, object]) -> list[str]:
    cmd: list[str] = [
        str(PYTHON_EXE),
        "-m",
        "scripts.icbr_benchmark",
        "--profile",
        str(cfg["profile"]),
        "--tasks",
        ",".join(str(x) for x in cfg["tasks"]),
        "--seeds",
        ",".join(str(x) for x in cfg["seeds"]),
        "--feynman-root",
        str(cfg["feynman_root"]),
        "--feynman-variant",
        str(cfg["feynman_variant"]),
        "--feynman-width-mid",
        str(cfg["feynman_width_mid"]),
        "--feynman-split-strategy",
        str(cfg["feynman_split_strategy"]),
        "--feynman-split-strategy-seed",
        str(cfg["feynman_split_strategy_seed"]),
        "--feynman-dataset-select-seed",
        str(cfg["feynman_dataset_select_seed"]),
        "--train-num",
        str(cfg["train_num"]),
        "--test-num",
        str(cfg["test_num"]),
        "--train-steps",
        str(cfg["train_steps"]),
        "--lr",
        str(cfg["lr"]),
        "--lamb",
        str(cfg["lamb"]),
        "--prune-iters",
        str(cfg["prune_iters"]),
        "--teacher-cache-dir",
        str(cfg["teacher_cache_dir"]),
        "--teacher-cache-mode",
        str(cfg["teacher_cache_mode"]),
        "--teacher-cache-version",
        str(cfg["teacher_cache_version"]),
        "--output-dir",
        str(cfg["output_dir"]),
        "--teacher-max-test-mse",
        str(cfg["teacher_max_test_mse"]),
        "--teacher-min-test-r2",
        str(cfg["teacher_min_test_r2"]),
        "--feynman-post-prune-steps",
        str(cfg["feynman_post_prune_steps"]),
        "--feynman-post-prune-lr",
        str(cfg["feynman_post_prune_lr"]),
        "--feynman-post-prune-lamb",
        str(cfg["feynman_post_prune_lamb"]),
        "--feynman-post-prune-eval-every",
        str(cfg["feynman_post_prune_eval_every"]),
        "--feynman-post-prune-min-delta",
        str(cfg["feynman_post_prune_min_delta"]),
        "--feynman-post-prune-patience",
        str(cfg["feynman_post_prune_patience"]),
    ]
    if cfg.get("feynman_fit_opt"):
        cmd.extend(["--feynman-fit-opt", str(cfg["feynman_fit_opt"])])
    if bool(cfg.get("quiet", False)):
        cmd.append("--quiet")
    return cmd


command = build_benchmark_command(CONFIG)
print(" ".join(command))

result = subprocess.run(
    command,
    cwd=REPO_ROOT,
    check=False,
    capture_output=True,
    text=True,
)

print(f"exit_code={result.returncode}")
if result.stdout.strip():
    print("\n[stdout]\n")
    print(result.stdout[-6000:])
if result.stderr.strip():
    print("\n[stderr]\n")
    print(result.stderr[-6000:])

if result.returncode != 0:
    raise RuntimeError("Benchmark command failed. See stderr above.")

c:\Users\chenpeng\miniconda3\envs\kan\python.exe -m scripts.icbr_benchmark --profile feynman_reference --tasks feynman_I_12_4 --seeds 1 --feynman-root datasets --feynman-variant Feynman_with_units --feynman-width-mid [5,2] --feynman-split-strategy random --feynman-split-strategy-seed 1 --feynman-dataset-select-seed 1 --train-num 2000 --test-num 1000 --train-steps 200 --lr 0.01 --lamb 0.01 --prune-iters 3 --teacher-cache-dir outputs/teacher_cache_feynman_tune --teacher-cache-mode off --teacher-cache-version feynman_tune_v1 --output-dir outputs/icbr_benchmark_feynman_tune_example --teacher-max-test-mse 0.1 --teacher-min-test-r2 0.0 --feynman-post-prune-steps 100 --feynman-post-prune-lr 0.001 --feynman-post-prune-lamb 0.001 --feynman-post-prune-eval-every 5 --feynman-post-prune-min-delta 1e-06 --feynman-post-prune-patience 2 --feynman-fit-opt Adam


## Inspect Teacher Quality and Cache Signals

Reads `icbr_benchmark_summary.json` and `icbr_benchmark_rows.csv` from the output directory.

In [ ]:
output_dir = (REPO_ROOT / str(CONFIG["output_dir"]).strip()).resolve()
summary_path = output_dir / "icbr_benchmark_summary.json"
rows_path = output_dir / "icbr_benchmark_rows.csv"

if not summary_path.is_file():
    raise FileNotFoundError(f"Missing summary JSON: {summary_path}")
if not rows_path.is_file():
    raise FileNotFoundError(f"Missing rows CSV: {rows_path}")

summary = json.loads(summary_path.read_text(encoding="utf-8"))

print("Summary files:")
print(f"- {summary_path}")
print(f"- {rows_path}")
print()
cfg_summary = summary.get("config", {})
teacher_training_cfg = cfg_summary.get("teacher_training", {})

print("Global training config snapshot:")
print({
    "train_num": cfg_summary.get("train_num"),
    "test_num": cfg_summary.get("test_num"),
    "train_steps": cfg_summary.get("train_steps"),
    "lr": cfg_summary.get("lr"),
    "lamb": cfg_summary.get("lamb"),
})

print("\nTeacher training config snapshot:")
for task_name, task_cfg in teacher_training_cfg.items():
    if task_name in CONFIG["tasks"]:
        print(task_name, {
            "train_steps": task_cfg.get("train_steps", cfg_summary.get("train_steps")),
            "lr": task_cfg.get("lr", cfg_summary.get("lr")),
            "lamb": task_cfg.get("lamb", cfg_summary.get("lamb")),
            "post_prune_steps": task_cfg.get("post_prune_steps"),
            "post_prune_lr": task_cfg.get("post_prune_lr"),
            "post_prune_lamb": task_cfg.get("post_prune_lamb"),
            "post_prune_eval_every": task_cfg.get("post_prune_eval_every"),
            "post_prune_patience": task_cfg.get("post_prune_patience"),
            "available_keys": sorted(task_cfg.keys()),
        })

def _safe_float(value: object) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("nan")

print("\nPer-row teacher metrics:")
with rows_path.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    selected = [
        {
            "task": r.get("task", ""),
            "seed": int(r.get("seed", -1)),
            "teacher_test_mse": _safe_float(r.get("teacher_test_mse")),
            "teacher_test_r2": _safe_float(r.get("teacher_test_r2")),
            "teacher_quality_gate_pass": r.get("teacher_quality_gate_pass", ""),
            "teacher_quality_gate_reason": r.get("teacher_quality_gate_reason", ""),
            "teacher_cache_hit": r.get("teacher_cache_hit", ""),
            "teacher_cache_status": r.get("teacher_cache_status", ""),
        }
        for r in reader
        if r.get("task") in CONFIG["tasks"]
    ]

for row in selected:
    print(row)

Summary files:
- D:\chenpeng\Documents\学习\数学\毕业论文\symkan-experiments\outputs\icbr_benchmark_feynman_tune_example\icbr_benchmark_summary.json
- D:\chenpeng\Documents\学习\数学\毕业论文\symkan-experiments\outputs\icbr_benchmark_feynman_tune_example\icbr_benchmark_rows.csv

Global training config snapshot:
{'train_num': 2000, 'test_num': 1000, 'train_steps': 200, 'lr': 0.01, 'lamb': 0.01}

Teacher training config snapshot:
feynman_I_13_4 {'train_steps': 200, 'lr': 0.01, 'lamb': 0.01, 'post_prune_steps': 100, 'post_prune_lr': 0.001, 'post_prune_lamb': 0.001, 'post_prune_eval_every': 5, 'post_prune_patience': 2, 'available_keys': ['grid', 'k', 'opt', 'post_prune_early_stop', 'post_prune_eval_every', 'post_prune_lamb', 'post_prune_lr', 'post_prune_min_delta', 'post_prune_patience', 'post_prune_steps', 'post_train_prune', 'prune_edge_th', 'prune_iters', 'prune_node_th']}

Per-row teacher metrics:
{'task': 'feynman_I_13_4', 'seed': 1, 'teacher_test_mse': 28.397314071655273, 'teacher_test_r2': 0.959352

## Reset / Cleanup Block (Safe)

Use this block between tuning rounds to reduce state carry-over.

- Always clears in-memory objects if present.
- File deletion is **off by default**.
- To delete temp outputs/cache for this notebook run, set `DELETE_FILES = True`.
- Deletion is safety-limited to repo `outputs/` and `tmp/` subpaths only.

In [ ]:
IN_MEMORY_VARS = [
    "result",
    "summary",
    "selected",
]
DELETE_FILES = False

for name in IN_MEMORY_VARS:
    if name in globals():
        del globals()[name]

gc.collect()
if torch is not None and torch.cuda.is_available():
    torch.cuda.empty_cache()

print("In-memory state cleaned.")

if DELETE_FILES:
    allow_roots = [
        (REPO_ROOT / "outputs").resolve(),
        (REPO_ROOT / "tmp").resolve(),
    ]
    target_paths = [
        (REPO_ROOT / str(CONFIG["output_dir"])).resolve(),
        (REPO_ROOT / str(CONFIG["teacher_cache_dir"])).resolve(),
    ]
    for path in target_paths:
        in_allow_root = any(str(path).startswith(str(root)) for root in allow_roots)
        if not in_allow_root:
            raise RuntimeError(f"Refuse to delete path outside outputs/tmp: {path}")
        if path.exists():
            shutil.rmtree(path)
            print(f"Deleted: {path}")
        else:
            print(f"Skip missing: {path}")
else:
    print("File cleanup skipped (DELETE_FILES=False).")

In-memory state cleaned.
File cleanup skipped (DELETE_FILES=False).


## Suggested Tuning Loop

1. Adjust `CONFIG` (especially `train_steps`, `lr`, `lamb`, and post-prune overrides).
2. Run benchmark cell.
3. Inspect teacher gate and cache rows.
4. Run reset/cleanup block before next round.
5. Send the best config back for script-level default calibration.